In [ ]:
import time
import uuid

from langchain_core.messages import HumanMessage
from langgraph.checkpoint.memory import MemorySaver
from opik.integrations.langchain import OpikTracer

from sahiloan_chatbot import logger
from sahiloan_chatbot.application.agent.workflow import ChatState, create_chat_graph
from sahiloan_chatbot.application.agent.workflow.schema import UserSchema

checkpointer = MemorySaver()

graph_builder = create_chat_graph()
graph = graph_builder.compile(checkpointer)
opik_tracer = OpikTracer(graph=graph.get_graph(xray=True), project_name="sahiloan_chatbot")
graph

In [ ]:
def get_response(user_query, thread_id=None):
    response = None
    logger.info(f"Agent Started | Query: {user_query}")
    start = time.perf_counter()

    agent_state = ChatState(
        user = UserSchema(user_id="22222222-2222-2222-2222-222222222222", username="shubham", first_name="Shubham", last_name="Prajapati"),
        messages=[HumanMessage(content=user_query)])

    if thread_id is None:
        thread_id = str(uuid.uuid4())[:8]
        logger.info(f"created new thread: {thread_id}")

    logger.info(f"Agent State: {agent_state}")
    config = {"configurable": {"thread_id": str(thread_id)}, "callbacks": [opik_tracer]}
    response = graph.invoke(agent_state, config=config)
    latency = (time.perf_counter() - start) * 1000
    logger.info(f"Agent Completed | Latency: {latency:.2f}ms")
    logger.info(f"Completed State: \n{response}")
    return response.get("messages")[-1].content

## General Query Agent

In [ ]:
get_response("which products sahiloan offers?")

'Sahiloan offers assistance with the following types of loans:\n\n- Home Loans\n- Loan Against Property (LAP)\n- Balance Transfer & Top-Up\n\nWe also have special programs for NRIs, self-employed individuals, and low documentation cases. If you have any specific questions about these products, feel free to ask!'

In [ ]:
get_response("how sahiloan will help me?")

"Sahiloan can help you in several ways:\n\n1. **Loan Comparison**: We assist you in comparing different lenders' interest rates and terms to ensure you get the best deal.\n2. **Verification of Offers**: If you've received an offer from a bank, we can help verify if it's genuinely good and identify any hidden conditions.\n3. **Support Throughout the Process**: We stay with you even after loan sanction, assisting with disbursement delays, EMI-related issues, interest rate revisions, and balance transfers.\n4. **Unbiased Advice**: Our service is completely free for borrowers, and we provide guidance without any hidden agendas or urgency tactics.\n\nThink of us as your trusted advisor throughout your entire loan journey!"



In [ ]:
get_response("I need car loan instantly")


"I understand that you're looking for an instant car loan. While I don't have specific information on instant loan options, I recommend visiting our website or contacting our customer service team for assistance. They can guide you through the application process and help you find the best loan options available for your needs. If you have any other questions or need further assistance, feel free to ask!"

## Loan Agent

In [ ]:
get_response("What's the current interest rate for my Home loans?")

'It looks like you have an active Home Loan with the following details:\n\n- **Lender Name**: HDFC\n- **Loan Amount**: ₹60,00,000\n- **Current Outstanding Balance**: ₹48,00,000\n- **Interest Rate**: 8.65%\n- **Tenure**: 240 months (20 years)\n- **EMI Amount**: ₹52,000\n- **Open Date**: June 10, 2020\n- **Due Date**: June 10, 2040\n\nIf you have any more questions or need further assistance, feel free to ask!'

In [ ]:
get_response("Tell me about my personal loan?")

"It looks like you have an active Personal Loan with the following details:\n\n- **Lender Name**: Kotak Mahindra\n- **Loan Amount**: ₹8,00,000\n- **Current Outstanding Balance**: ₹4,20,000\n- **Interest Rate**: 13.5%\n- **Tenure**: 48 months\n- **EMI Amount**: ₹21,500\n- **Open Date**: February 10, 2023\n- **Due Date**: February 10, 2027\n\nIf you have any more questions or need further assistance, feel free to ask!"


In [ ]:
get_response("List down all my loans")

'You have three active loans:\n\n1. **Home Loan**\n   - **Lender**: HDFC\n   - **Loan Amount**: ₹60,00,000\n   - **Outstanding Balance**: ₹48,00,000\n   - **Interest Rate**: 8.65%\n   - **EMI Amount**: ₹52,000\n   - **Tenure**: 240 months (20 years)\n   - **Open Date**: June 10, 2020\n   - **Due Date**: June 10, 2040\n\n2. **Loan Against Property**\n   - **Lender**: Axis Bank\n   - **Loan Amount**: ₹50,00,000\n   - **Outstanding Balance**: ₹35,00,000\n   - **Interest Rate**: 9.75%\n   - **EMI Amount**: ₹52,000\n   - **Tenure**: 180 months (15 years)\n   - **Open Date**: August 12, 2022\n   - **Due Date**: August 12, 2037\n\n3. **Personal Loan**\n   - **Lender**: Kotak Mahindra\n   - **Loan Amount**: ₹8,00,000\n   - **Outstanding Balance**: ₹4,20,000\n   - **Interest Rate**: 13.5%\n   - **EMI Amount**: ₹21,500\n   - **Tenure**: 48 months (4 years)\n   - **Open Date**: February 10, 2023\n   - **Due Date**: February 10, 2027\n\nIf you have any specific questions about any of these loans, feel free to ask!'

In [ ]:
## Todos

# Q. What is the current interest rate for my home loan?
# Q. Tell me about my home loan?
# Q. How many loans I have taken?
# Q. What is my current outstanding balance on the home loan?
# Q. Whats the current interest rate for my loan againt property?
# Q. How many EMIs have I already paid for my SBI home loan?
# Q. What is my current outstanding balance on the home loan?

# Uses the following tools:
# - get_user_loans_tool
